In [1]:
# GPU Programming with PyTorch
# Demonstrating CPU vs GPU execution, tensor movement,
# and workload optimization

import time
import torch
import torch.nn as nn
import torch.optim as optim

# --------------------------------------------------
# 1. Check GPU Availability
# --------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

if torch.cuda.is_available():
    print(
        "GPU Name:",
        torch.cuda.get_device_name(0)
    )
    print(
        "CUDA Version:",
        torch.version.cuda
    )


Using device: cpu


In [3]:

# --------------------------------------------------
# 2. Create Tensors on CPU and GPU
# --------------------------------------------------

cpu_tensor = torch.randn(3000, 3000)

if torch.cuda.is_available():

    gpu_tensor = torch.randn(
        3000,
        3000,
        device=device
    )

    print(
        "CPU Tensor Device:",
        cpu_tensor.device
    )

    print(
        "GPU Tensor Device:",
        gpu_tensor.device
    )


In [4]:

# --------------------------------------------------
# 3. Compare CPU and GPU Matrix Multiplication
# --------------------------------------------------

def time_matrix_multiplication(device_name):

    x = torch.randn(
        3000,
        3000,
        device=device_name
    )

    y = torch.randn(
        3000,
        3000,
        device=device_name
    )

    if device_name == "cuda":
        torch.cuda.synchronize()

    start_time = time.time()

    result = torch.matmul(x, y)

    if device_name == "cuda":
        torch.cuda.synchronize()

    end_time = time.time()

    return end_time - start_time


cpu_time = time_matrix_multiplication("cpu")

print(
    f"CPU Matrix Multiplication Time: "
    f"{cpu_time:.4f} seconds"
)

if torch.cuda.is_available():

    gpu_time = time_matrix_multiplication("cuda")

    print(
        f"GPU Matrix Multiplication Time: "
        f"{gpu_time:.4f} seconds"
    )

    print(
        f"Speedup: "
        f"{cpu_time / gpu_time:.2f}x"
    )



CPU Matrix Multiplication Time: 0.1802 seconds


In [5]:
# --------------------------------------------------
# 4. Move Tensors Between CPU and GPU
# --------------------------------------------------

x_cpu = torch.randn(5, 5)

print(
    "Original Device:",
    x_cpu.device
)

x_gpu = x_cpu.to(device)

print(
    "Moved Device:",
    x_gpu.device
)

x_back_to_cpu = x_gpu.cpu()

print(
    "Back to CPU:",
    x_back_to_cpu.device
)



Original Device: cpu
Moved Device: cpu
Back to CPU: cpu


In [6]:
# --------------------------------------------------
# 5. Build a Simple Neural Network
# --------------------------------------------------

class SimpleGPUModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(784, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )

    def forward(self, x):

        return self.network(x)


model = SimpleGPUModel().to(device)

print(model)

print(
    "Model Device:",
    next(model.parameters()).device
)


SimpleGPUModel(
  (network): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=10, bias=True)
  )
)
Model Device: cpu


In [10]:

# --------------------------------------------------
# 6. Create Synthetic Training Data
# --------------------------------------------------

num_samples = 10000
input_size = 784
num_classes = 10

X = torch.randn(
    num_samples,
    input_size
)

y = torch.randint(
    0,
    num_classes,
    (num_samples,)
)

X = X.to(device)
y = y.to(device)

print(
    "Input Device:",
    X.device
)

print(
    "Label Device:",
    y.device
)


Input Device: cpu
Label Device: cpu


In [11]:

# --------------------------------------------------
# 7. Train Model on Selected Device
# --------------------------------------------------

loss_function = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

num_epochs = 5
batch_size = 128

for epoch in range(num_epochs):

    model.train()

    total_loss = 0

    for start in range(
        0,
        num_samples,
        batch_size
    ):

        X_batch = X[
            start:start + batch_size
        ]

        y_batch = y[
            start:start + batch_size
        ]

        outputs = model(X_batch)

        loss = loss_function(
            outputs,
            y_batch
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    average_loss = (
        total_loss /
        (num_samples // batch_size)
    )

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Loss: {average_loss:.4f}"
    )


Epoch 1/5 | Loss: 2.3394
Epoch 2/5 | Loss: 2.1891
Epoch 3/5 | Loss: 1.7106
Epoch 4/5 | Loss: 1.0269
Epoch 5/5 | Loss: 0.3512


In [12]:

# --------------------------------------------------
# 8. Evaluate Model Without Gradients
# --------------------------------------------------

model.eval()

with torch.no_grad():

    outputs = model(X)

    predictions = torch.argmax(
        outputs,
        dim=1
    )

    accuracy = (
        (predictions == y)
        .float()
        .mean()
        * 100
    )

print(
    f"Training Accuracy: "
    f"{accuracy:.2f}%"
)


Training Accuracy: 99.46%


In [17]:

# --------------------------------------------------
# 9. Monitor GPU Memory Usage
# --------------------------------------------------

if torch.cuda.is_available():

    allocated = (
        torch.cuda.memory_allocated()
        / 1024**2
    )

    reserved = (
        torch.cuda.memory_reserved()
        / 1024**2
    )

    print(
        f"GPU Memory Allocated: "
        f"{allocated:.2f} MB"
    )

    print(
        f"GPU Memory Reserved: "
        f"{reserved:.2f} MB"
    )


In [16]:

# --------------------------------------------------
# 10. Clear Unused GPU Memory
# --------------------------------------------------

if torch.cuda.is_available():

    torch.cuda.empty_cache()

    allocated_after = (
        torch.cuda.memory_allocated()
        / 1024**2
    )

    reserved_after = (
        torch.cuda.memory_reserved()
        / 1024**2
    )

    print(
        f"Memory Allocated After Cache Clear: "
        f"{allocated_after:.2f} MB"
    )

    print(
        f"Memory Reserved After Cache Clear: "
        f"{reserved_after:.2f} MB"
    )


In [9]:

# --------------------------------------------------
# 11. Recommended GPU Usage Pattern
# --------------------------------------------------

def get_device():

    return torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )


def move_batch_to_device(
    batch_inputs,
    batch_labels,
    device
):

    batch_inputs = batch_inputs.to(device)
    batch_labels = batch_labels.to(device)

    return batch_inputs, batch_labels


device = get_device()

print(
    "Recommended Device:",
    device
)

Recommended Device: cpu
